# 2. Feature Engineering

Extract KPIs, embeddings, and process statistics.

## 2.0 Install Dependencies

In [1]:
# Install required packages (safe to re-run; --quiet suppresses noise)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
    'pandas',
    'pyarrow',
    'numpy',
    'scikit-learn',
    'joblib',
    'matplotlib',
    'seaborn',
    'sentence-transformers',
])


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


0

## 2.1 Setup

In [2]:
import sys; from pathlib import Path
_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / 'src' / 'data_ingestion.py').exists():
        _repo_root = _p
        _src = str(_p / 'src')
        if _src not in sys.path: sys.path.insert(0, _src)
        break
DATASET = "BPIC2012"
OUTPUT_DIR = _repo_root / 'output' / DATASET
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2.1 Load Data

In [3]:
import pandas as pd
df = pd.read_parquet(OUTPUT_DIR / f'events_{DATASET}_train.parquet')
print(f"Loaded {len(df)} events")

Loaded 262200 events


## 2.2 Compute KPIs

In [4]:
from feature_engineering import compute_case_kpis
kpis = compute_case_kpis(df)
kpis.to_parquet(OUTPUT_DIR / f'kpis_{DATASET}_train.parquet', engine='pyarrow', index=False)
print('KPIs saved')

KPIs saved


## 2.3 Transition Matrix

In [5]:
from feature_engineering import compute_transition_matrix
trans_mat = compute_transition_matrix(df)
trans_mat.to_parquet(OUTPUT_DIR / f'transition_matrix_{DATASET}_train.parquet', engine='pyarrow')
print('Transition matrix saved')

Transition matrix saved


## 2.4 Activity Embeddings

In [6]:
from feature_engineering import ActivityEmbedder
import joblib
embedder = ActivityEmbedder(vector_size=32, seed=42)
embedder.fit(df)
joblib.dump(embedder, OUTPUT_DIR / f'activity_embeddings.model')
print('Embeddings saved')

Embeddings saved


## 2.5 Terminal Classification

Classifies which activities represent bad terminal states (rejection, cancellation)
using semantic sentence embeddings (`all-MiniLM-L6-v2`). Runs once here and
saves the result so `ProcessEnv` can load it instantly without re-downloading
the model on every instantiation.

In [7]:
from feature_engineering import classify_bad_terminals
import json

activities = sorted(df['activity'].unique().tolist())
bad_terminals = classify_bad_terminals(activities)

terminal_info = {
    'activities':     activities,
    'bad_terminals':  sorted(bad_terminals),
}
with open(OUTPUT_DIR / 'terminal_classification.json', 'w') as f:
    json.dump(terminal_info, f, indent=2)

print(f'Bad terminals: {sorted(bad_terminals)}')
print(f'Saved to {OUTPUT_DIR / "terminal_classification.json"}')

/mnt/hdd/Code/Git/bwa/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2126.35it/s]


Bad terminals: ['A_CANCELLED', 'A_DECLINED', 'O_CANCELLED', 'O_DECLINED']
Saved to /mnt/hdd/Code/Git/bwa/output/BPIC2012/terminal_classification.json


## 2.5 Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# KPI distributions
kpis = pd.read_parquet(OUTPUT_DIR / f'kpis_{DATASET}_train.parquet')
axes[0, 0].hist(kpis['case_age_days'], bins=30, color='steelblue', edgecolor='black')
axes[0, 0].set_xlabel('Case Age (days)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Throughput Time Distribution')

# Trace length distribution (replaces waiting_time which is not in KPI output)
axes[0, 1].hist(kpis['trace_length'], bins=30, color='coral', edgecolor='black')
axes[0, 1].set_xlabel('Trace Length (events)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Trace Length Distribution')

# Transition matrix heatmap
trans_mat = pd.read_parquet(OUTPUT_DIR / f'transition_matrix_{DATASET}_train.parquet', engine='pyarrow')
if not trans_mat.empty:
    sns.heatmap(trans_mat, ax=axes[1, 0], cmap='Blues', annot=False)
    axes[1, 0].set_title('Transition Matrix Heatmap')
    axes[1, 0].tick_params(axis='both', labelsize=7)
else:
    axes[1, 0].text(0.5, 0.5, 'Transition matrix empty', ha='center', va='center')
    axes[1, 0].set_title('Transition Matrix')

# Activity count
activity_counts = df['activity'].value_counts()
axes[1, 1].pie(activity_counts.head(8).values, labels=activity_counts.head(8).index, autopct='%1.1f%%', startangle=90)
axes[1, 1].set_title('Top 8 Activities (Pie Chart)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_engineering_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {OUTPUT_DIR / 'feature_engineering_overview.png'}")